In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import torch
from datasets import Dataset
from setfit import SetFitModel
from setfit import SetFitTrainer


In [ ]:
df = pd.read_csv("Test_Data.csv",index_col=False).drop_duplicates("text")
df["label"] = df["label"].astype(str).str.strip().str.lower()

#split model into train and test and cv - stratified 

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

#encode labels

label2id = {"down": 0, "neutral": 1, "up": 2}
id2label = {v: k for k, v in label2id.items()}

for df_part in [train_df, val_df, test_df]:
    df_part["label"] = df_part["label"].map(label2id)

#convert to hugging face datasets


train_ds = Dataset.from_pandas(train_df[["text", "label"]])
val_ds = Dataset.from_pandas(val_df[["text", "label"]])
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

#load Setfit

model = SetFitModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

#Train Setfit


trainer = SetFitTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric="f1",
    batch_size=16,
    num_iterations=20,   # how many contrastive steps
    num_epochs=1         # classifier training
)

trainer.train()


#eval model

preds = trainer.model(test_ds["text"])
true = test_ds["label"]

print("Accuracy:", accuracy_score(true, preds))
print("Macro F1:", f1_score(true, preds, average="macro"))

def weighted_sentiment(probs,entropy):

    """
    probs: array of shape (n_sentences, 3)
        columns = [p_up, p_neutral, p_down]
    """

    #extract probs
    p_up = probs[:, 2]
    p_neutral = probs[:, 1]
    p_down = probs[:, 0]

    #calculate score
    scores = p_up - p_down

    #normalise entropy
    H = entropy
    H_norm = H / np.log(3.0)

    #calculate weighted score 
    weights = np.abs(p_up - p_down) * (1 - H_norm)

    if np.sum(weights) == 0:
            return 0.0

    return np.sum(weights * scores) / np.sum(weights)

#build rates expectations index - Ei = P_up - P_down, 

def rates_expectations(model, texts, labels=("down","neutral","up")):

    """
    Computes rate expectation scores from text using a SetFit model.

    Returns:
        dict with:
        - scores: per-text (p_up - p_down)
        - mean_score: aggregate expectation
        - probs: full probability matrix
        - predictions: predicted labels
        - confidence: confidence of predicted class
    """


    # Get probabilities
    probs = model.predict_proba(texts)

    #convert to numpy array 
    if isinstance(probs, torch.Tensor):
        probs = probs.detach().cpu().numpy()
    else:
        probs = np.asarray(probs)

    # Predictions + confidence
    pred_idx = np.argmax(probs, axis=1)
    predictions = [labels[i] for i in pred_idx]
    confidence = probs[np.arange(len(probs)), pred_idx]
    entropy = -np.sum(probs* np.log(probs + 1e-10), axis=1)

    values_dict = {
        "weighted_score": weighted_sentiment(probs, entropy),
        "entropy": entropy,
        "probs": probs,
        "predictions": predictions,
        "confidence": confidence
     }


    return weighted_sentiment(probs, entropy), values_dict


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
C:\Users\joven\AppData\Local\Temp\ipykernel_63424\2242069295.py:36: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(


Map:   0%|          | 0/105 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 4200
  Batch size = 16
  Num epochs = 1
c:\Users\joven\miniconda3\envs\setfit\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.324900
50,0.231200
100,0.156800
150,0.080800
200,0.034000
250,0.021000


Accuracy: 0.782608695652174
Macro F1: 0.7805555555555556


In [14]:
tests = [
    "Inflation is expected to rise sharply next quarter.",
    "Cooling demand and falling energy prices should reduce inflation.",
    "The central bank left rates unchanged while analysts remain divided."
]
sentiment, dict = rates_expectations(model, tests)

print(sentiment)
print(dict["entropy"])
print(dict["predictions"])

0.003684445747132376
[0.28340142 0.28044296 0.33858721]
['up', 'down', 'neutral']
